# 04 Virtual Directed Evolution

This notebook follows the project requirement: Round 0 uses the existing training data as historical experimental results; Round 1 runs the first recommendation and evaluates it with hidden true fitness from the test set; Round 2 uses Round 1 feedback to recommend again; Round 3 uses feedback from the first two rounds to recommend again. It compares random mutation, direct fitness-model recommendation, LLM Agent recommendation, and knowledge-enhanced LLM Agent recommendation.

In [ ]:
from pathlib import Path

import pandas as pd

from knowledge_base import apply_mutation_count_constraint, score_mutation_rules, weighted_knowledge_score
from llm_agent_core import add_mutation_features, build_designer_pool, load_two_vs_many, summarize_history
from report_figures import summarize_rounds
from virtual_evolution import (
    select_model_top_candidates,
    select_random_candidates,
    simulate_iterative_evolution,
    summarize_recommended_positions,
)

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
ROUNDS = 3
TOP_K = 10
MAX_MUTATIONS_FOR_KNOWLEDGE = 4


In [ ]:
observed_df, candidate_pool = load_two_vs_many("two_vs_many.csv")
observed_features = add_mutation_features(observed_df)

test_truth = pd.read_csv("test.csv").reset_index(drop=True)
test_truth["candidate_id"] = [f"C{i:05d}" for i in range(len(test_truth))]
test_candidates = add_mutation_features(test_truth)

top_history, mutation_summary = summarize_history(observed_df)
designer_pool = build_designer_pool(candidate_pool, mutation_summary, pool_size=5000)
truth_pool = designer_pool.merge(
    test_candidates[["candidate_id", "target"]],
    on="candidate_id",
    how="left",
)

prediction_path = ARTIFACTS / "test_esm2_mlp_predictions.csv"
if prediction_path.exists():
    predictions = pd.read_csv(prediction_path).reset_index(drop=True)
    predictions["candidate_id"] = [f"C{i:05d}" for i in range(len(predictions))]
    predictions = predictions.rename(columns={"predicted_fitness": "esm_predicted_fitness"})
    truth_pool = truth_pool.merge(
        predictions[["candidate_id", "esm_predicted_fitness"]],
        on="candidate_id",
        how="left",
    )
else:
    truth_pool["esm_predicted_fitness"] = truth_pool["historical_prior"]

truth_pool["predicted_fitness"] = truth_pool["esm_predicted_fitness"].fillna(truth_pool["historical_prior"])
truth_pool["prediction_error"] = truth_pool["predicted_fitness"] - truth_pool["target"]

rule_scores = truth_pool["mutations"].apply(score_mutation_rules).apply(pd.Series)
truth_pool = pd.concat([truth_pool.reset_index(drop=True), rule_scores.reset_index(drop=True)], axis=1)
truth_pool = weighted_knowledge_score(truth_pool, historical_weight=0.7, rule_weight=0.3)

print("Initial observed training variants:", len(observed_features))
print("Hidden test candidates for virtual evaluation:", len(truth_pool))
display(top_history[["mutated_region", "target", "mutations", "num_mutations"]].head(10))
display(truth_pool[[
    "candidate_id",
    "mutations",
    "num_mutations",
    "target",
    "predicted_fitness",
    "prediction_error",
    "historical_prior",
    "rule_score",
    "knowledge_enhanced_score",
]].head())


In [ ]:
agent_path = ARTIFACTS / "llm_agent_final_recommendations.csv"
agent_seed_ids = []
if agent_path.exists():
    agent_seed_ids = pd.read_csv(agent_path)["candidate_id"].astype(str).tolist()
agent_seed_ids = list(dict.fromkeys(agent_seed_ids))

def score_with_current_history(current_observed, available_candidates):
    _, current_mutation_summary = summarize_history(current_observed, top_history_count=200)
    scored = build_designer_pool(
        available_candidates[["sequence", "candidate_id"]],
        current_mutation_summary,
        pool_size=len(available_candidates),
    )
    scored = scored.drop(columns=["target", "predicted_fitness", "prediction_error"], errors="ignore")
    keep_cols = [
        "candidate_id",
        "target",
        "esm_predicted_fitness",
        "predicted_fitness",
        "prediction_error",
        "knowledge_enhanced_score",
    ]
    return scored.merge(available_candidates[keep_cols], on="candidate_id", how="left")

def random_strategy(current_observed, available_candidates, round_index, top_k):
    return select_random_candidates(available_candidates["candidate_id"], top_k=top_k, seed=round_index)

def model_strategy(current_observed, available_candidates, round_index, top_k):
    return select_model_top_candidates(available_candidates, top_k=top_k, score_col="predicted_fitness")

def llm_agent_strategy(current_observed, available_candidates, round_index, top_k):
    dynamic_pool = score_with_current_history(current_observed, available_candidates)
    seeded = dynamic_pool[dynamic_pool["candidate_id"].isin(agent_seed_ids)].copy()
    seeded_ids = select_model_top_candidates(seeded, top_k=min(top_k, len(seeded)), score_col="historical_prior") if not seeded.empty else []
    if len(seeded_ids) >= top_k:
        return seeded_ids[:top_k]
    fallback_pool = dynamic_pool[~dynamic_pool["candidate_id"].isin(seeded_ids)]
    fallback_ids = select_model_top_candidates(fallback_pool, top_k=top_k - len(seeded_ids), score_col="historical_prior")
    return seeded_ids + fallback_ids

def knowledge_enhanced_strategy(current_observed, available_candidates, round_index, top_k):
    dynamic_pool = score_with_current_history(current_observed, available_candidates)
    constrained = apply_mutation_count_constraint(dynamic_pool, max_mutations=MAX_MUTATIONS_FOR_KNOWLEDGE)
    if constrained.empty:
        constrained = dynamic_pool
    rule_scores = constrained["mutations"].apply(score_mutation_rules).apply(pd.Series)
    constrained = pd.concat([constrained.reset_index(drop=True), rule_scores.reset_index(drop=True)], axis=1)
    constrained = weighted_knowledge_score(constrained, historical_weight=0.7, rule_weight=0.3)
    return select_model_top_candidates(constrained, top_k=top_k, score_col="knowledge_enhanced_score")

strategies = {
    "random_mutation": random_strategy,
    "fitness_model_direct": model_strategy,
    "llm_agent": llm_agent_strategy,
    "knowledge_enhanced_llm_agent": knowledge_enhanced_strategy,
}

results = simulate_iterative_evolution(
    observed_features,
    truth_pool,
    strategies,
    rounds=ROUNDS,
    top_k=TOP_K,
    evaluator_col="target",
)
results.to_csv(ARTIFACTS / "virtual_evolution_results.csv", index=False)

recommended_topk = results[results["round"] > 0].copy()
recommended_topk.to_csv(ARTIFACTS / "virtual_evolution_topk_by_round.csv", index=False)

display(recommended_topk[[
    "strategy",
    "round",
    "selection_rank",
    "candidate_id",
    "mutations",
    "num_mutations",
    "predicted_fitness",
    "fitness",
    "prediction_error",
    "round_best_fitness",
    "observed_count_before_round",
]])


In [ ]:
summary = summarize_rounds(recommended_topk)
summary["mean_fitness_change_vs_previous_round"] = summary.groupby("strategy")["mean_fitness"].diff()
summary["best_fitness_change_vs_previous_round"] = summary.groupby("strategy")["best_fitness"].diff()
summary["mean_fitness_improved_vs_previous_round"] = summary["mean_fitness_change_vs_previous_round"].fillna(0) > 0
summary["best_fitness_improved_vs_previous_round"] = summary["best_fitness_change_vs_previous_round"].fillna(0) > 0

summary.to_csv(ARTIFACTS / "virtual_evolution_summary.csv", index=False)
display(summary)

position_summary = summarize_recommended_positions(recommended_topk)
position_summary.to_csv(ARTIFACTS / "virtual_evolution_key_position_summary.csv", index=False)
top_positions = (
    position_summary
    .sort_values(["strategy", "round", "mutation_frequency", "mutation_count"], ascending=[True, True, False, False])
    .groupby(["strategy", "round"])
    .head(8)
    .reset_index(drop=True)
)
display(top_positions)

ax = summary.pivot(index="round", columns="strategy", values="best_fitness").plot(marker="o", figsize=(8, 4))
ax.set_ylabel("Best true fitness in recommended Top-k")
ax.set_title("Best recommendation fitness over 3 virtual rounds")
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(ARTIFACTS / "virtual_evolution_curve.png", dpi=200)

mean_ax = summary.pivot(index="round", columns="strategy", values="mean_fitness").plot(marker="o", figsize=(8, 4))
mean_ax.set_ylabel("Mean true fitness of recommended Top-k")
mean_ax.set_title("Mean recommendation fitness over 3 virtual rounds")
mean_fig = mean_ax.get_figure()
mean_fig.tight_layout()
mean_fig.savefig(ARTIFACTS / "virtual_evolution_mean_curve.png", dpi=200)

pos_ax = top_positions.pivot_table(
    index="position",
    columns="strategy",
    values="mutation_frequency",
    aggfunc="max",
).fillna(0).plot(kind="bar", figsize=(9, 4))
pos_ax.set_ylabel("Max mutation frequency in a Top-k round")
pos_ax.set_title("Recommended mutations concentrate on key positions")
pos_fig = pos_ax.get_figure()
pos_fig.tight_layout()
pos_fig.savefig(ARTIFACTS / "virtual_evolution_key_positions.png", dpi=200)
